In [ ]:
from presidio_evaluator import InputSample

train_samples = InputSample.read_dataset_json("../data/train_May-04-2026.json")
test_samples = InputSample.read_dataset_json("../data/test_May-04-2026.json")

print(f"train: {len(train_samples)} 筆")
print(f"test: {len(test_samples)} 筆")

# 檢查前幾筆有沒有正常解析
for s in train_samples[:2]:
    print(s.full_text)
    print(s.tags)
    print("---")

In [ ]:
InputSample.create_spacy_dataset(train_samples, output_path="train_en.spacy")
InputSample.create_spacy_dataset(test_samples, output_path="dev_en.spacy")

In [ ]:
cd D:\2.programm2\github-star\presidio-research\app
python -m spacy init config config_en.cfg --lang en --pipeline ner --optimize accuracy --force
python -m spacy train config_en.cfg --output ./output --paths.train ./train_en.spacy --paths.dev ./dev_en.spacy
@REM python -m spacy train config_en.cfg --output ./output_en --paths.train ./train.spacy --paths.dev ./dev.spacy --training.max_steps 200 --training.eval_frequency 20 --training.patience 0

In [ ]:
E    #       LOSS TOK2VEC  LOSS NER  ENTS_F  ENTS_P  ENTS_R  SCORE 
---  ------  ------------  --------  ------  ------  ------  ------
  0       0          0.00     56.00    0.00    0.00    0.00    0.00
  1     200         75.07   1784.19   59.63   61.38   57.98    0.60
  2     400         37.20    657.89   67.03   75.51   60.26    0.67
  3     600        717.07    430.43   67.59   71.79   63.84    0.68
  5     800         55.44    267.19   66.99   65.93   68.08    0.67
  8    1000         33.59    111.28   57.95   58.92   57.00    0.58
 11    1200         31.68     67.29   60.22   58.64   61.89    0.60
 15    1400         43.32     62.20   67.61   65.64   69.71    0.68
 19    1600         22.90     29.68   63.49   64.12   62.87    0.63
 25    1800        129.83    110.31   65.91   65.70   66.12    0.66
 32    2000         89.92     79.72   70.13   77.78   63.84    0.70
 40    2200          6.64      4.56   66.43   74.49   59.93    0.66
 50    2400         36.36     32.47   70.10   74.18   66.45    0.70
 61    2600         16.93     15.23   66.12   66.12   66.12    0.66
 71    2800         86.66     61.33   64.45   65.76   63.19    0.64
 82    3000       1763.96    298.90   67.86   75.10   61.89    0.68
 92    3200        117.08    103.38   69.23   74.72   64.50    0.69
103    3400         61.56     38.48   72.09   73.56   70.68    0.72
113    3600         44.24     25.99   68.27   79.31   59.93    0.68
124    3800        252.34    106.47   71.53   78.82   65.47    0.72
134    4000         84.58     46.24   71.28   78.21   65.47    0.71
145    4200         39.48      4.73   70.59   75.28   66.45    0.71
155    4400         16.29      7.87   71.81   80.00   65.15    0.72
166    4600         97.12     38.11   66.89   69.10   64.82    0.67
176    4800        220.38     70.63   65.26   71.15   60.26    0.65
187    5000        231.09     81.63   68.38   67.41   69.38    0.68
✔ Saved pipeline to output directory

In [4]:
import spacy

nlp = spacy.load("./output/model-best")

print(nlp.get_pipe("ner").labels)

('DATE', 'GPE', 'NORP', 'ORG', 'PERSON')


In [ ]:
import spacy

nlp = spacy.load("./output/model-best")

text = "Justin sent his email test@example.com to Kat in 6/8/2023 and she replied to him on 6/9/2023. They also discussed the new project at 10:00 AM."
doc = nlp(text)

for ent in doc.ents:
    print(ent.text, "->", ent.label_)

Justin -> PERSON
Kat -> ORG
6/8/2023 -> DATE
6/9/2023 -> DATE
10:00 -> DATE


In [9]:
import time
from datetime import datetime

import pyperclip
import spacy

# -----------------------------
# Load custom spaCy model
# -----------------------------
MODEL_PATH = "./output/model-best"

print("=" * 60)
print("Loading custom model...")
nlp = spacy.load(MODEL_PATH)
print("Model loaded successfully.")
print("NER Labels:", nlp.get_pipe("ner").labels)
print("=" * 60)

last_text = ""


def anonymize(text, entities):
    """
    Replace entities with their labels.
    """
    result = text

    # Replace from back to front to avoid index shifting
    for ent in sorted(entities, key=lambda x: x.start_char, reverse=True):
        replacement = f"<{ent.label_}>"
        result = (
            result[:ent.start_char]
            + replacement
            + result[ent.end_char:]
        )

    return result


print("Clipboard monitor started...")
print("Press Ctrl+C to stop.\n")

while True:

    try:
        current = pyperclip.paste()

        if current != last_text and current.strip():

            last_text = current

            print("\n" + "=" * 70)
            print("Clipboard Updated")
            print("=" * 70)
            print("Time :", datetime.now().strftime("%Y-%m-%d %H:%M:%S"))
            print()

            print("Original Text")
            print("-" * 70)
            print(current)
            print()

            start = time.perf_counter()

            doc = nlp(current)

            elapsed = (time.perf_counter() - start) * 1000

            print("Detected Entities")
            print("-" * 70)

            if len(doc.ents) == 0:
                print("No entities detected.")

            else:

                statistics = {}

                for i, ent in enumerate(doc.ents, start=1):

                    statistics[ent.label_] = statistics.get(ent.label_, 0) + 1

                    print(f"Entity #{i}")
                    print(f"Text       : {ent.text}")
                    print(f"Label      : {ent.label_}")
                    print(f"Start      : {ent.start_char}")
                    print(f"End        : {ent.end_char}")
                    print()

                print("-" * 70)
                print("Statistics")

                for label, count in statistics.items():
                    print(f"{label:10} : {count}")

            print()

            anonymized = anonymize(current, doc.ents)

            print("-" * 70)
            print("Anonymized Text")
            print("-" * 70)
            print(anonymized)
            print()

            pyperclip.copy(anonymized)

            print("Clipboard updated successfully.")
            print(f"NER Processing Time : {elapsed:.2f} ms")
            print("=" * 70)

        time.sleep(0.3)

    except KeyboardInterrupt:
        print("\nProgram stopped.")
        break

    except Exception as e:
        print(e)
        time.sleep(1)

Loading custom model...
Model loaded successfully.
NER Labels: ('DATE', 'GPE', 'NORP', 'ORG', 'PERSON')
Clipboard monitor started...
Press Ctrl+C to stop.


Clipboard Updated
Time : 2026-07-05 11:53:08

Original Text
----------------------------------------------------------------------
import time
from datetime import datetime

import pyperclip
import spacy

# -----------------------------
# Load custom spaCy model
# -----------------------------
MODEL_PATH = "./output/model-best"

print("=" * 60)
print("Loading custom model...")
nlp = spacy.load(MODEL_PATH)
print("Model loaded successfully.")
print("NER Labels:", nlp.get_pipe("ner").labels)
print("=" * 60)

last_text = ""


def anonymize(text, entities):
    """
    Replace entities with their labels.
    """
    result = text

    # Replace from back to front to avoid index shifting
    for ent in sorted(entities, key=lambda x: x.start_char, reverse=True):
        replacement = f"<{ent.label_}>"
        result = (
            result